In [43]:
import requests
import time

URL = "https://dataportal.livsmedelsverket.se/livsmedel/api/v1"

def get_nutritions(limit=200, sprak=1, delay=0.1):
    
    url = f"{URL}/livsmedel"
    offset = 0
    nutritions = []

    while True:
        params = {
            "offset": offset,
            "limit": limit,
            "sprak": sprak
        }

        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
        except requests.exceptions.RequestException as err:
            print(f"Offset error: {err}")
            break

        try:
            data = response.json()
        except ValueError as err:
            print(f"JSON error: {err}")

        nutrition_data = data.get("livsmedel", [])
        if not isinstance(nutrition_data, list):
            print(f"API call in wrong format")

        nutritions.extend(nutrition_data)

        if len(nutrition_data) < limit:
            break

        offset += limit
        time.sleep(delay)

    return nutritions


In [44]:
nutritions = get_nutritions()

print("Totalt hämtat:", len(nutritions))

Totalt hämtat: 2575


In [42]:
print(nutritions[2]["namn"])
print(nutritions[2]["nummer"])
print(nutritions[2]["links"])


Gris ister
3
[{'href': '/api/v1/livsmedel/3/naringsvarden?sprak=1', 'rel': 'naringvarden', 'method': 'GET'}, {'href': '/api/v1/livsmedel/3/klassificeringar?sprak=1', 'rel': 'klassificeringar', 'method': 'GET'}, {'href': '/api/v1/livsmedel/3/ravaror?sprak=1', 'rel': 'ravaror', 'method': 'GET'}]


In [ ]:
import json
print(json.dumps(nutritions[0], indent=2))

In [ ]:
import requests
import time
import pandas as pd


URL = "https://dataportal.livsmedelsverket.se/livsmedel/api/v1"

def get_nutrition_values(nummer, sprak=1):
    url = f"{URL}/livsmedel/{nummer}/naringsvarden"
    params = {"sprak": sprak}

    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
    except requests.exceptions.RequestException as err:
        print(f"Error getting values for {nummer}: {err}")
        return []

    try:
        data = resp.json()
    except ValueError as err:
        print(f"JSON error: {err}")

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        return data.get("naringsvarden", [])

    print(f"Wrong format for values in {nummer}")
    return []
    url = f"{URL}/livsmedel/{nummer}/klassificeringar?sprak={sprak}"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        group_data = response.json()
        
        for item in group_data:
            if item.get("typ") == "Huvudgrupp":
                return item.get("kod", "Övrigt")
                
        return "Övrigt"
    
    except requests.exceptions.RequestException as e:
        print(f"Fel vid hämtning av livsmedel {nummer}: {e}")
        return "Okänd"
    except ValueError as e:  # JSON-dekodningsfel
        print(f"Fel vid tolkning av JSON för livsmedel {nummer}: {e}")
        return "Okänd"
    except Exception as e:
        print(f"Oväntat fel för livsmedel {nummer}: {e}")
        return "Okänd"

def create_dataframe(sprak=1):
    try:
        products = get_nutritions(sprak=sprak)
    except Exception as e:
        print(f"Couldn't get product list: {e}")
        return pd.DataFrame

    records = []

    for item in products:
        nummer = item["nummer"]
        namn = item["namn"]

        value_list = get_nutrition_values(nummer, sprak=sprak)

        for value in value_list:
            records.append({
                "nummer": nummer,
                "namn": namn,
                "naringsnamn": value.get("namn"),
                "mangd": value.get("varde"),
                "enhet": value.get("enhet")
            })

        time.sleep(0.05)

    df = pd.DataFrame(records)

    return df

In [24]:
df = create_dataframe()
print(df.head())
# print(len(df))

Hämtade 200 st (offset 0)
Hämtade 200 st (offset 200)
Hämtade 200 st (offset 400)
Hämtade 200 st (offset 600)
Hämtade 200 st (offset 800)
Hämtade 200 st (offset 1000)
Hämtade 200 st (offset 1200)
Hämtade 200 st (offset 1400)
Hämtade 200 st (offset 1600)
Hämtade 200 st (offset 1800)
Hämtade 200 st (offset 2000)
Hämtade 200 st (offset 2200)
Hämtade 175 st (offset 2400)
  → Hämtar näringsvärden för 'Nöt talg' (1)
  → Hämtar näringsvärden för 'Gris späck' (2)
  → Hämtar näringsvärden för 'Gris ister' (3)
  → Hämtar näringsvärden för 'Kokosfett' (4)
  → Hämtar näringsvärden för 'Matfettsblandning havssaltat fett 80% berikad typ Bregott' (5)
  → Hämtar näringsvärden för 'Matfettsblandning fett 60% berikad typ Bregott mellan' (6)
  → Hämtar näringsvärden för 'Flytande margarin fett 82% berikad typ Milda culinesse' (10)
  → Hämtar näringsvärden för 'Hushållsmargarin fett 80% berikad typ Melba' (12)
  → Hämtar näringsvärden för 'Hushållsmargarin fett 80% berikad typ Milda' (13)
  → Hämtar närin

In [33]:
df.head(5)

,nummer,namn,naringsnamn,mangd,enhet
0,1,Nöt talg,"Zink, Zn",0.1,mg
1,1,Nöt talg,Vitamin E,0.6,mg
2,1,Nöt talg,Vitamin D,0.1,µg
3,1,Nöt talg,Vitamin C,0.0,mg
4,1,Nöt talg,Vitamin B6,0.0,mg


In [67]:
import requests
import json

BASE_URL = "https://dataportal.livsmedelsverket.se/livsmedel/api/v1"
sprak = 1  # svenska

# Hämta t.ex. de första 5 livsmedlen
response = requests.get(f"{BASE_URL}/livsmedel?limit=5&sprak={sprak}")
data = response.json()

# Skriv ut hela JSON-strukturen snyggt
print(json.dumps(data, indent=2))



{
  "_meta": {
    "totalRecords": 2575,
    "offset": 0,
    "limit": 5,
    "count": 5
  },
  "_links": [
    {
      "href": "/api/v1/livsmedel?offset=0&limit=5&sprak=1",
      "rel": "self",
      "method": "GET"
    },
    {
      "href": "/api/v1/livsmedel?offset=0&limit=5&sprak=1",
      "rel": "first",
      "method": "GET"
    },
    {
      "href": "/api/v1/livsmedel?offset=2570&limit=5&sprak=1",
      "rel": "last",
      "method": "GET"
    },
    {
      "href": "/api/v1/livsmedel?offset=5&limit=5&sprak=1",
      "rel": "next",
      "method": "GET"
    }
  ],
  "livsmedel": [
    {
      "livsmedelsTypId": 1,
      "livsmedelsTyp": "Analyserat",
      "nummer": 1,
      "version": "2025-10-07T14:09:29.22",
      "namn": "N\u00f6t talg",
      "vetenskapligtNamn": "Bos taurus",
      "projekt": "2023 Uppdatering av laktosv\u00e4rden",
      "links": [
        {
          "href": "/api/v1/livsmedel/1/naringsvarden?sprak=1",
          "rel": "naringvarden",
          "method

In [75]:
livsmedel_id = 1  # byt till den produkt du vill kolla
klass_url = f"{BASE_URL}/livsmedel/{livsmedel_id}/klassificeringar?sprak={sprak}"
klass_data = requests.get(klass_url).json()

print(json.dumps(klass_data, indent=2))


[
  {
    "typ": "LanguaL",
    "fasett": "A Gruppindelning EuroFIR",
    "fasettkod": "A0777, A0352",
    "namn": "Andra djurfetter",
    "langualId": "A0810"
  },
  {
    "typ": "LanguaL",
    "fasett": "B Artklassificering",
    "fasettkod": "B1564",
    "namn": "N\u00f6tkreatur",
    "langualId": "B1161"
  },
  {
    "typ": "LanguaL",
    "fasett": "C Del av v\u00e4xt/djur",
    "fasettkod": "C0116",
    "namn": "Fett eller olja",
    "langualId": "C0190"
  },
  {
    "typ": "LanguaL",
    "fasett": "E Fysisk form",
    "fasettkod": "E0113",
    "namn": "Halvfast med mjuk konsistens, (mosad, bredbar)",
    "langualId": "E0119"
  },
  {
    "typ": "LanguaL",
    "fasett": "F V\u00e4rmebehandling",
    "fasettkod": "F0011",
    "namn": "Ej v\u00e4rmebehandlad (r\u00e5)",
    "langualId": "F0003"
  },
  {
    "typ": "LanguaL",
    "fasett": "G Tillagning",
    "fasettkod": "G0002",
    "namn": "Tillagningsmetod ej till\u00e4mplig",
    "langualId": "G0003"
  },
  {
    "typ": "LanguaL

In [76]:
gruppering = None
for item in klass_data:
    if item.get("typ") == "Huvudgrupp":
        gruppering = item.get("kod")
        break
gruppering

'Övrigt fett (ister, talg, kokosfett)'

In [ ]:
def food_group(nummer, sprak=1):
    url = f"{URL}/livsmedel/{nummer}/klassificeringar?sprak={sprak}"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        group_data = response.json()
        
        for item in group_data:
            if item.get("typ") == "Huvudgrupp":
                return item.get("kod", "Övrigt")
                
        return "Övrigt"
    
    except requests.exceptions.RequestException as e:
        print(f"Fel vid hämtning av livsmedel {nummer}: {e}")
        return "Okänd"
    except ValueError as e:
        print(f"Fel vid tolkning av JSON för livsmedel {nummer}: {e}")
        return "Okänd"
    except Exception as e:
        print(f"Oväntat fel för livsmedel {nummer}: {e}")
        return "Okänd"

In [ ]:
import requests
import pandas as pd
import time

BASE_URL = "https://dataportal.livsmedelsverket.se/livsmedel/api/v1"
SPRAK = 1
LIMIT = 200

def get_all_livsmedel():
    offset = 0
    all_livsmedel = []

    while True:
        url = f"{BASE_URL}/livsmedel?limit={LIMIT}&offset={offset}&sprak={SPRAK}"
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Fel vid hämtning offset {offset}: {response.status_code}")
            break
        data = response.json()
        livsmedel_lista = data.get("livsmedel", [])
        if not livsmedel_lista:
            break
        all_livsmedel.extend(livsmedel_lista)
        offset += LIMIT
        time.sleep(0.1)

    return all_livsmedel

# Hämta alla livsmedel
livsmedel_lista = get_all_livsmedel()
print(f"Totalt livsmedel hämtade: {len(livsmedel_lista)}")

# Loop för att extrahera Gruppering
results = []
for item in livsmedel_lista:
    nummer = item['nummer']
    gruppering = food_group(nummer)
    results.append({
        "nummer": nummer,
        "namn": item['namn'],
        "Gruppering": gruppering
    })

df = pd.DataFrame(results)
print(df.head())

In [82]:
antal_unika = df['Gruppering'].nunique()
print(antal_unika)

118


In [83]:
unik_gruppering = df['Gruppering'].unique()
print(unik_gruppering)

['Övrigt fett (ister, talg, kokosfett)' 'Hård matfettsblandning'
 'Flytande matfettsblandning' 'Sås dressing majonnäs ' 'Smör' 'Olja'
 'Majonnässallad röror' 'Sallad blandad mat' 'Mesvaror' 'Färskost o kvarg'
 'Hård ost mm' 'Dessertost' 'Smältost' 'Osträtter' 'Mjölk'
 'Naturell fil yoghurt'
 'Mjölkdryck chokladdryck milkshake smothie m yoghurt'
 'Smaksatt fil yoghurt' 'Hårt bröd ' 'Mjukt bröd ' 'Mjöl stärkelse kli'
 'Riskakor' 'Potatis' 'Potatisprodukter potatisrätter'
 'Köttprodukter kötträtter' 'Fisk o skaldjursprodukter o rätter'
 'Soppa mat' 'Rotfrukter' 'Grönsaksjuice rotfruktsjuice'
 'Grönsaks- rotfrukts- baljväxträtter o produkter' 'Grönsaker'
 'Baljväxter (bönor, linser och ärter)' 'Svamp'
 'Grönsaksblandningar med rotfrukter och eller baljväxter' 'Pastarätter'
 'Frukt färsk fryst' 'Bär färska frysta' 'Buljong' 'Gelatin agar agar'
 'Vegetabiliskt protein produkter och rätter' 'Frukt o bär torkade'
 'Söta soppor kräm o efterrättssås' 'Frukt o bär konserverade'
 'Fruktjuice mm' '

In [88]:
df[df['Gruppering'] == "Pasta"]

,nummer,namn,Gruppering
633,819,Nudlar glasnudlar okokta,Pasta
653,845,Pasta okokt,Pasta
654,846,Pasta kokt u. salt,Pasta
655,848,Pasta fullkorn okokt,Pasta
656,852,Pasta färsk m. ägg kokt u. salt,Pasta
665,870,Nudlar äggnudlar okokta,Pasta
1638,2079,Nudlar äggnudlar kokta m. salt,Pasta
1658,2125,Nudlar glasnudlar kokta m. salt,Pasta
1687,2220,Pasta makaroner spagetti okokt glutenfri,Pasta
1688,2221,Pasta fusilli lasagneplattor okokt glutenfri,Pasta


In [ ]:
def get_food_group(nummer, sprak=1):
    url = f"{URL}/livsmedel/{nummer}/klassificeringar"
    params = {"sprak": sprak}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Error getting values for {nummer}: {e}")

    for item in data:
        if item.get("typ") == "Huvudgrupp":
            return item.get("kod", "Övrigt")
    
    return "Övrigt"

test_nummer = [5665, 1234, 230]  # några exempelnummer
for n in test_nummer:
    print(f"{n}: {get_food_group(n)}")

5665: Pastarätter
1234: Övrigt
230: Potatis


In [ ]:
def get_food_group(nummer, sprak=1):
    url = f"{URL}/livsmedel/{nummer}/klassificeringar"
    params = {"sprak": sprak}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        print(f"Debug {nummer}: {data}")  # <-- här ser du hela JSON-svaret
    except Exception as e:
        print(f"Error getting values for {nummer}: {e}")
        return "Okänd"

    for item in data:
        if item.get("typ") == "Huvudgrupp":
            return item.get("kod", "Övrigt")
    
    return "Övrigt"

In [95]:
import requests
import pandas as pd

URL = "https://dataportal.livsmedelsverket.se/livsmedel/api/v1"

def get_nutritions(limit=10, sprak=1):
    url = f"{URL}/livsmedel"
    params = {"offset": 0, "limit": limit, "sprak": sprak}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data.get("livsmedel", [])
    except Exception as e:
        print(f"Error fetching nutritions: {e}")
        return []

def get_food_group(nummer, sprak=1):
    url = f"{URL}/livsmedel/{nummer}/klassificeringar"
    params = {"sprak": sprak}
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Error getting food group for {nummer}: {e}")
        return "Övrigt"

    for item in data:
        if item.get("typ") == "Huvudgrupp":
            return item.get("kod", "Övrigt")
    return "Övrigt"

# Hämta 10 produkter
products = get_nutritions(limit=10)

results = []
for item in products:
    nummer = item["nummer"]
    namn = item["namn"]
    group = get_food_group(nummer)
    results.append({
        "nummer": nummer,
        "namn": namn,
        "Gruppering": group
    })

df_test = pd.DataFrame(results)
print(df_test)

   nummer                                               namn  \
0       1                                           Nöt talg   
1       2                                         Gris späck   
2       3                                         Gris ister   
3       4                                          Kokosfett   
4       5  Matfettsblandning havssaltat fett 80% berikad ...   
5       6  Matfettsblandning fett 60% berikad typ Bregott...   
6      10  Flytande margarin fett 82% berikad typ Milda c...   
7      12        Hushållsmargarin fett 80% berikad typ Melba   
8      13        Hushållsmargarin fett 80% berikad typ Milda   
9      17            Lättmargarin fett 38% berikad typ Becel   

                             Gruppering  
0  Övrigt fett (ister, talg, kokosfett)  
1  Övrigt fett (ister, talg, kokosfett)  
2  Övrigt fett (ister, talg, kokosfett)  
3  Övrigt fett (ister, talg, kokosfett)  
4                Hård matfettsblandning  
5                Hård matfettsblandning  
6